1) Method Choice and Why 

Method Choice: Random Forest Classifier
Why: Random Forest is an ensemble learning method that constructs multiple decision trees and merges them together. I chose this because it is highly accurate, handles non-linear relationships in SEO metrics (like position vs. CTR) effectively, and is far more resistant to overfitting than a single decision tree.

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# 1. Load baseline queue if output exists, or generate rule labels dynamically
if os.path.exists('../../outputs/baseline_action_score.csv'):
    baseline_df = pd.read_csv('../../outputs/baseline_action_score.csv')
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
    df['action_label'] = 'None'
    df.loc[df['content_id'].isin(baseline_df['content_id']), 'action_label'] = 'CTR-fix'
else:
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
    PAGE_1_THRESHOLD = 10
    BAD_CTR_THRESHOLD = 0.01 
    rule_mask = (df['avg_position'] <= PAGE_1_THRESHOLD) & (df['ctr'] < BAD_CTR_THRESHOLD)
    df['action_label'] = 'None'
    df.loc[rule_mask, 'action_label'] = 'CTR-fix'

# 2. Define Features (X)
X = df[['avg_position', 'ctr', 'impressions_90d']]

# 3. Define Target (y) - Convert 'CTR-fix' label into a binary 1 (yes) or 0 (no)
y = (df['action_label'] == 'CTR-fix').astype(int)

# 4. The Split Design
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print verification results
print(f"Training data shape (rows, columns): {X_train.shape}")
print(f"Testing data shape (rows, columns): {X_test.shape}")
print("Split design executed successfully. The test data is securely locked away.")

Training data shape (rows, columns): (24000, 3)
Testing data shape (rows, columns): (6000, 3)
Split design executed successfully. The test data is securely locked away.


In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Fallback setup if running cell independently
if 'X_train' not in locals() and 'X_train' not in globals():
    if os.path.exists('../../outputs/baseline_action_score.csv'):
        baseline_df = pd.read_csv('../../outputs/baseline_action_score.csv')
        df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
        df['action_label'] = 'None'
        df.loc[df['content_id'].isin(baseline_df['content_id']), 'action_label'] = 'CTR-fix'
    else:
        df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
        PAGE_1_THRESHOLD = 10
        BAD_CTR_THRESHOLD = 0.01 
        rule_mask = (df['avg_position'] <= PAGE_1_THRESHOLD) & (df['ctr'] < BAD_CTR_THRESHOLD)
        df['action_label'] = 'None'
        df.loc[rule_mask, 'action_label'] = 'CTR-fix'

    X = df[['avg_position', 'ctr', 'impressions_90d']]
    y = (df['action_label'] == 'CTR-fix').astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Initialize the Random Forest model
# max_depth=5 keeps the trees relatively simple so they don't overfit
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 2. Train (fit) the model on the 80% Training Set
rf_model.fit(X_train, y_train)

# 3. Make predictions on the 20% Testing Set
rf_predictions = rf_model.predict(X_test)

# 4. Recreate your Week 4 Baseline on the exact same test data for a fair fight
# Rule from last week: Position <= 10 AND CTR < 0.01
baseline_predictions = ((X_test['avg_position'] <= 10) & (X_test['ctr'] < 0.01)).astype(int)

# 5. Print the Scoreboard!
print("--- THE SCOREBOARD ---")
print(f"Week 4 Baseline Rule Accuracy: {accuracy_score(y_test, baseline_predictions):.4f}")
print(f"Week 5 Random Forest Accuracy: {accuracy_score(y_test, rf_predictions):.4f}")

print("\n--- Detailed Model Report ---")
print(classification_report(y_test, rf_predictions))

--- THE SCOREBOARD ---
Week 4 Baseline Rule Accuracy: 1.0000
Week 5 Random Forest Accuracy: 1.0000

--- Detailed Model Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4829
           1       1.00      1.00      1.00      1171

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



Errors and Interpretation:
The Random Forest model achieved a perfect accuracy of 1.0, with zero false positives and zero false negatives. In a real-world scenario, a perfect score on behavioral data strongly indicates a deterministic target or target leakage. Because my Week 4 Baseline Rule (avg_position <= 10 and ctr < 0.01) also achieved a 1.0 accuracy, it proves the action_label in this dataset was procedurally generated using that exact mathematical logic. The model did not learn nuanced SEO user behavior; it simply reverse-engineered the hard-coded baseline formula because it was fed the exact features (avg_position and ctr) used to create the target.